In [7]:
import pandas as pd
import time
from astroquery.ipac.ned import Ned
import astropy.units as u
from astropy.coordinates import SkyCoord
import warnings

In [8]:
# 忽略一些不必要的警告
warnings.filterwarnings('ignore', category=UserWarning)

In [11]:
# 1. 配置路径和参数
cat_path = '/home/jclou/loujc_work/sofia/GAMA/1300-1350/Dec-0011_09_05_arcdrift/Dec-0011_09_05_arcdrift_cat.txt'
top_n = 268  # 自动检索信噪比最高的前 N 个源
search_radius = 1.5 * u.arcmin # 搜索半径
f_rest = 1420.40575 # HI 静止频率 (MHz)

In [12]:
# 2. 自动加载 SoFiA 目录
columns = ["name", "id", "x", "y", "z", "x_min", "x_max", "y_min", "y_max", "z_min", "z_max", 
           "n_pix", "f_min", "f_max", "f_sum", "rel", "flag", "rms", "w20", "w50", "wm50", 
           "z_w20", "z_w50", "z_wm50", "ell_maj", "ell_min", "ell_pa", "ell3s_maj", "ell3s_min", 
           "ell3s_pa", "kin_pa", "err_x", "err_y", "err_z", "err_f_sum", "snr", "snr_max", 
           "ra", "dec", "freq", "x_peak", "y_peak", "z_peak", "ra_peak", "dec_peak", "freq_peak"]

df = pd.read_csv(cat_path, sep='\s+', comment='#', names=columns)

# 按信噪比排序，取前 Top N 个进行验证
df_top = df.sort_values('snr', ascending=False).head(top_n).reset_index(drop=True)
print(f"成功加载目录，共有 {len(df)} 个源。即将对 SNR 最高的 {top_n} 个源进行自动检索...\n")
results = []

成功加载目录，共有 268 个源。即将对 SNR 最高的 268 个源进行自动检索...



In [13]:
# 3. 自动循环检索
for i, row in df_top.iterrows():
    s_id = int(row['id'])
    ra, dec = row['ra'], row['dec']
    f_obs = row['freq'] / 1e6  # 将 Hz 转为 MHz
    z_hi = (f_rest / f_obs) - 1
    
    print(f"[{i+1}/{top_n}] 正在检索 Source ID {s_id} (z_hi: {z_hi:.4f})...", end="\r")
    
    try:
        co = SkyCoord(ra=ra, dec=dec, unit=(u.deg, u.deg))
        # 执行 NED 查询
        res = Ned.query_region(co, radius=search_radius)
        
        if res:
            res_df = res.to_pandas()
            # 寻找红移匹配的源 (误差容限 delta_z < 0.002)
            matches = res_df[abs(res_df['Redshift'] - z_hi) < 0.002]
            
            if not matches.empty:
                obj_name = matches.iloc[0]['Object Name']
                obj_z = matches.iloc[0]['Redshift']
                results.append({
                    'ID': s_id, 'SNR': row['snr'], 'RA': ra, 'Dec': dec,
                    'z_HI': z_hi, 'NED_Match': obj_name, 'z_NED': obj_z, 'Status': '✅ 匹配'
                })
            else:
                results.append({
                    'ID': s_id, 'SNR': row['snr'], 'RA': ra, 'Dec': dec,
                    'z_HI': z_hi, 'NED_Match': 'N/A', 'z_NED': None, 'Status': '❓ 无红移匹配'
                })
        else:
            results.append({
                'ID': s_id, 'SNR': row['snr'], 'RA': ra, 'Dec': dec,
                'z_HI': z_hi, 'NED_Match': 'N/A', 'z_NED': None, 'Status': '❌ 无天体'
            })
            
    except Exception:
        results.append({
            'ID': s_id, 'SNR': row['snr'], 'RA': ra, 'Dec': dec,
                    'z_HI': z_hi, 'NED_Match': 'Error', 'z_NED': None, 'Status': '⚠️ 查询超时'
        })
    
    # 适当休眠，防止请求过快被 NED 封禁
    time.sleep(0.5)

In [ ]:
# 4. 展示最终对比表
print("\n" + "="*50)
print("检索完成！结果汇总如下：")
# 设置显示的最大行数，设为 None 表示不限制
pd.set_option('display.max_rows', None)
# 设置显示的最大列宽，防止文字被截断
pd.set_option('display.max_colwidth', None)
summary_df = pd.DataFrame(results)
display(summary_df)


检索完成！结果汇总如下：


,ID,SNR,RA,Dec,z_HI,NED_Match,z_NED,Status
0,4,214.011723,204.820861,-0.133595,0.067918,N/A,NaN,❓ 无红移匹配
1,1,188.261263,187.478256,-0.130474,0.067822,N/A,NaN,❓ 无红移匹配
2,6,177.931737,134.288009,-0.163298,0.067998,N/A,NaN,❓ 无红移匹配
3,2,144.345453,152.671019,-0.146775,0.067669,N/A,NaN,❓ 无红移匹配
4,3,139.766231,175.734447,-0.131856,0.067974,N/A,NaN,❓ 无红移匹配
...,...,...,...,...,...,...,...,...
263,141,2.779284,140.312410,0.029790,0.069518,N/A,NaN,❓ 无红移匹配
264,147,2.739834,145.271209,-0.173630,0.069760,N/A,NaN,❓ 无红移匹配
265,166,2.736665,141.640037,0.027793,0.072822,N/A,NaN,❓ 无红移匹配
266,240,2.707112,162.281735,0.022014,0.080419,N/A,NaN,❓ 无红移匹配
